# Bronze — Statistics Iceland GDP Ingestion

Fetches quarterly GDP year-on-year growth from Statistics Iceland (Hagstofa Íslands) via the PX-Web REST API.

**Source:** Hagstofa Íslands PX-Web API  
**Dataset:** THJ01601 — Quarterly GDP (Mælikvarði=2 YoY growth, Skipting=14 Total GDP)  
**Output:** `bronze.statistics_iceland_gdp` (Delta table)  
**Schedule:** Quarterly  

In [ ]:
import requests
import pandas as pd

API_URL = (
    "https://px.hagstofa.is/pxis/api/v1/is"
    "/Efnahagur/thjodhagsreikningar/landsframl"
    "/2_landsframleidsla_arsfj/THJ01601.px"
)
START_YEAR = 2022

payload = {
    "query": [
        {"code": "Mælikvarði", "selection": {"filter": "item", "values": ["2"]}},
        {"code": "Skipting",   "selection": {"filter": "item", "values": ["14"]}},
    ],
    "response": {"format": "json"},
}

response = requests.post(API_URL, json=payload)
response.raise_for_status()
data = response.json()

print(f"Total data points returned: {len(data['data'])}")

In [ ]:
records = []
for item in data["data"]:
    quarter = item["key"][2]
    raw_value = item["values"][0]
    records.append({
        "quarter": quarter,
        "year": int(quarter[:4]),
        "q": int(quarter[5]),
        "metric": "gdp_yoy_growth",
        "value": float(raw_value) if raw_value != ".." else None,
        "ingested_at": pd.Timestamp.now(),
    })

df_gdp = pd.DataFrame(records)
df_gdp = df_gdp[df_gdp["year"] >= START_YEAR]

if len(df_gdp) == 0:
    raise ValueError(f"[bronze_statistics] API returned 0 rows for year >= {START_YEAR} — possible API outage.")

# Incremental filter — only keep quarters not already in the table
if spark.catalog.tableExists("bronze.statistics_iceland_gdp"):
    row = spark.sql("""
        SELECT year, q FROM bronze.statistics_iceland_gdp
        ORDER BY year DESC, q DESC
        LIMIT 1
    """).collect()
    if row:
        last_year, last_q = row[0]["year"], row[0]["q"]
        df_gdp = df_gdp[
            (df_gdp["year"] > last_year) |
            ((df_gdp["year"] == last_year) & (df_gdp["q"] > last_q))
        ]
        print(f"Incremental load — {len(df_gdp)} new quarter(s) after {last_year}Q{last_q}")
    else:
        print(f"Full load — table empty, loading all {len(df_gdp)} rows")
else:
    print(f"Full load — {len(df_gdp)} rows")

print(df_gdp.to_string(index=False))

In [ ]:
if len(df_gdp) == 0:
    print("Table is already up to date. Nothing to write.")
else:
    spark_df = spark.createDataFrame(df_gdp)
    spark_df.write.format("delta").mode("append").saveAsTable("bronze.statistics_iceland_gdp")
    print(f"Appended {spark_df.count()} row(s) to bronze.statistics_iceland_gdp")